<a href="https://colab.research.google.com/github/KarthyV/payflow_finetuning_Ai_model/blob/main/payflow_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate datasets pyarrow

In [ ]:
pip install pyarrow==17.0.0 datasets==3.0.1

In [ ]:
import torch
import transformers
import peft
import trl
import datasets
import bitsandbytes

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"TRL: {trl.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Colab GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="payflow_faqs.jsonl")

In [ ]:
train_test_split = dataset['train'].train_test_split(test_size=0.2, seed = 42)

train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print("Total dataset length", len(dataset['train']))
print("Train dataset length", len(train_test_split['train']))
print("Test dataset length", len(train_test_split['test']))

In [ ]:
def formatting_func(example):
    return {"text": f"Question: {example['instruction']}\n\nAnswer: {example['output']}"}

train_dataset = train_dataset.map(formatting_func)
test_dataset = test_dataset.map(formatting_func)

# Verify it worked
print("First training example:")
print(train_dataset[0]['text'])
print("\n" + "="*80 + "\n")
print("Second training example:")
print(train_dataset[1]['text'])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# @title
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./payflow_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    save_steps=20,
    logging_steps=10,
    learning_rate=2e-4,
    bf16=True,
    weight_decay=0.01,
    warmup_steps=1
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args
)

In [ ]:
from google.colab import drive

trainer.train()

drive.mount('/content/drive')
save_path = '/content/drive/My Drive/payflow_model'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


In [ ]:
### Helper Function for generating responses

def generate_response(model, tokenizer, question, max_length):
  prompt = f"Question: {question}\n\nAnswer:"

  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

  outputs = model.generate(
      **inputs,
      max_length=max_length,
      temperature=0.7,
      top_p=0.9,
      do_sample=True
  )

  response = tokenizer.decode(outputs[0], skip_special_tokens=True)

  answer = response.split("Answer:")[-1].strip()

  return answer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading base model (unmodified TinyLlama)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

base_model= AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Base model loaded")

In [ ]:
from peft import PeftModel

print("Loading fine-tuned model (base + LoRA adapters)...")

ft_model = AutoModelForCausalLM.from_pretrained(
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    quantization_config=bnb_config,
    device_map="auto"
)

ft_model = PeftModel.from_pretrained(ft_model, './payflow_model/checkpoint-54')

print("✅ Fine-tuned model loaded\n")

In [ ]:
import random

random.seed(42)
sample_indices = random.sample(range(len(test_dataset)), 5)
test_samples = [test_dataset[i] for i in sample_indices]
print(test_samples)

In [ ]:
print("=" * 80)
print("COMPARISON: Base Model vs Fine-Tuned Model")
print("=" * 80 + "\n")

for idx, sample in enumerate(test_samples, 1):
    # Extract question
    question = sample["text"].split("\n\n")[0].replace("Question: ", "")

    print(f"{'─'*80}")
    print(f"FAQ #{idx}")
    print(f"{'─'*80}")
    print(f"Q: {question}\n")

    # Base model
    print("BASE MODEL Answer:")
    base_answer = generate_response(base_model, tokenizer, question, max_length=512)
    print(f"{base_answer}\n")

    # Fine-tuned model
    print("FINE-TUNED MODEL Answer:")
    ft_answer = generate_response(ft_model, tokenizer, question, max_length=512)
    print(f"{ft_answer}\n")

    print()

print("=" * 80)
print("✅ Evaluation complete!")
print("=" * 80)